# AI Music OS — Kokoro TTS (Colab-safe)

This notebook is designed for current Google Colab runtimes, including Python 3.13.

**Important:** the PyPI release `kokoro 0.9.4` currently declares Python `<3.13`, while the upstream repository has a Python 3.13-compatible change. Therefore this notebook uses the upstream Git repository when Colab is on Python 3.13 instead of forcing the incompatible PyPI release. citeturn1search1turn1search3


In [ ]:
# STEP 1 — Google Drive + system dependencies
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, subprocess, importlib.util

ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
CACHE = ROOT/'cache'/'huggingface'
OUT = ROOT/'outputs'/'kokoro'

for p in (CACHE, OUT):
    p.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE/'hub')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE/'transformers')

print('Python:', sys.version)
print('Python executable:', sys.executable)

# Kokoro needs espeak-ng for its fallback/phonemization path.
subprocess.run(
    ['apt-get','-qq','-y','install','espeak-ng'],
    check=True
)
print('espeak-ng: OK')


In [ ]:
# STEP 2 — Install Kokoro without the Python 3.13 PyPI trap
#
# Colab currently uses Python 3.13 in many runtimes.
# PyPI kokoro 0.9.4 declares Python <3.13, while upstream main has
# the Python 3.13 compatibility change.
#
# We also install from the upstream repositories together so Kokoro
# and Misaki stay on compatible code.

import subprocess, sys

py_minor = sys.version_info.minor

if sys.version_info >= (3, 13):
    print('Python 3.13+ detected -> installing current upstream Kokoro + Misaki')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-U',
        'git+https://github.com/hexgrad/misaki.git',
        'git+https://github.com/hexgrad/kokoro.git',
        'soundfile'
    ], check=True)
else:
    print('Python <=3.12 detected -> installing stable PyPI Kokoro')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-U',
        'kokoro>=0.9.4',
        'soundfile'
    ], check=True)

print('Kokoro installation completed.')


In [ ]:
# STEP 3 — Verify installation before starting the UI
import torch
import kokoro
from kokoro import KPipeline

print('Kokoro module:', kokoro.__file__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU is not available. Kokoro can run on CPU, but it will be slower.')

# Load the English/American pipeline.
pipeline = KPipeline(lang_code='a')
print('Kokoro pipeline: OK')


In [ ]:
# STEP 4 — Generate audio
import numpy as np
import soundfile as sf
import time

VOICE_CHOICES = [
    'af_heart','af_bella','af_nicole','af_sarah','af_sky',
    'am_adam','am_michael','bf_emma','bf_isabella',
    'bm_george','bm_lewis'
]

def generate_audio(text, voice='af_heart'):
    text = (text or '').strip()
    if not text:
        raise ValueError('Text is empty.')

    chunks = []
    sample_rate = 24000

    # Current Kokoro API yields tuples: (graphemes, phonemes, audio).
    # This replaces the incorrect dict-based handling from the previous ZIP.
    for item in pipeline(text, voice=voice):
        if isinstance(item, tuple) and len(item) >= 3:
            audio = item[2]
        elif isinstance(item, dict):
            audio = item.get('audio')
        else:
            audio = getattr(item, 'audio', None)

        if audio is None:
            raise RuntimeError(f'Unsupported Kokoro output type: {type(item)}')

        chunks.append(np.asarray(audio))

    if not chunks:
        raise RuntimeError('Kokoro returned no audio.')

    audio = np.concatenate(chunks)
    path = OUT / f'kokoro_{int(time.time())}.wav'
    sf.write(path, audio, sample_rate)

    return str(path)

test_file = generate_audio(
    'Hello. This is a test of the fixed AI Music OS Kokoro module.',
    'af_heart'
)
print('TEST PASSED:', test_file)


In [ ]:
# STEP 5 — Gradio UI
import gradio as gr

def ui_generate(text, voice):
    try:
        return generate_audio(text, voice)
    except Exception as e:
        raise gr.Error(str(e))

demo = gr.Interface(
    fn=ui_generate,
    inputs=[
        gr.Textbox(
            lines=8,
            label='Text',
            value='Hello. This is the AI Music OS Kokoro voice generator.'
        ),
        gr.Dropdown(
            choices=VOICE_CHOICES,
            value='af_heart',
            label='Voice'
        )
    ],
    outputs=gr.Audio(label='Generated audio', type='filepath'),
    title='AI Music OS — Kokoro TTS'
)

demo.launch(share=True, debug=True)
